In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-5")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bb382e49-ae3b-4a66-8b0a-89cd5cb178d6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 144ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

From orders.csv, select order_id, customer_id, order_date, and status using all three styles — string names, col(), and df["col"]. Confirm all three produce the same result.

In [2]:
from pyspark.sql.types import * 
from pyspark.sql import functions as F
orders=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv",header=True,inferSchema=True)
#method 1
orders.select("order_id","customer_id","order_date","status").show(4)
#method 2
orders.select(F.col("order_id"),F.col("customer_id"),F.col("order_date"),F.col("status")).show(4)
#method 3
orders.select(orders.order_id,orders.customer_id,orders.order_date,orders.status).show(4)


26/08/06 13:25:55 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|   O0001|       C001|2023-01-05|Delivered|
|   O0002|       C002|2023-01-07|Delivered|
|   O0003|       C003|2023-01-10|Delivered|
|   O0004|       C004|2023-01-12|Delivered|
+--------+-----------+----------+---------+
only showing top 4 rows


+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|   O0001|       C001|2023-01-05|Delivered|
|   O0002|       C002|2023-01-07|Delivered|
|   O0003|       C003|2023-01-10|Delivered|
|   O0004|       C004|2023-01-12|Delivered|
+--------+-----------+----------+---------+
only showing top 4 rows


+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|   O0001|       C001|2023-01-05|Delivered|
|   O0002|       C002|2023-01-07|Delivered|
|   O0003|       C003|2023-01-10|Delivered|
|   O0004|       C004|2023-01-12|Delivered|
+--------+-----------+----------+---------+
only showing top 4 rows


**Task 2**

Using select() with col(), compute a new column called revenue as unit_price * quantity and another called final_price as unit_price * (1 - discount_pct / 100).

 Show order_id, unit_price, quantity, discount_pct, revenue, and final_price.

In [3]:
orders.select(F.col('order_id'),
F.col('unit_price'),
F.col('quantity'),
(F.col('unit_price')*F.col('quantity')).alias('revenue'),
(F.col('unit_price')*(1-F.col('discount_pct')/100)).alias('final_price')).show(4)

+--------+----------+--------+-------+-----------+
|order_id|unit_price|quantity|revenue|final_price|
+--------+----------+--------+-------+-----------+
|   O0001|   1299.99|       2|2599.98|   1169.991|
|   O0002|    449.99|       1| 449.99|     449.99|
|   O0003|    349.99|       4|1399.96|   297.4915|
|   O0004|     89.99|       2| 179.98|    85.4905|
+--------+----------+--------+-------+-----------+
only showing top 4 rows


**Task 3**

Rewrite Task 2 using selectExpr() instead of select(). 

Add one more expression: UPPER(region) AS region_upper.

Compare the output.

In [4]:
orders.selectExpr("order_id",
"unit_price",
"quantity",
"unit_price*quantity as revenue",
"unit_price*(1-discount_pct/100) as final_price",
"upper(region) as region_upper").show(4)

+--------+----------+--------+-------+-----------+------------+
|order_id|unit_price|quantity|revenue|final_price|region_upper|
+--------+----------+--------+-------+-----------+------------+
|   O0001|   1299.99|       2|2599.98|   1169.991|        EAST|
|   O0002|    449.99|       1| 449.99|     449.99|        WEST|
|   O0003|    349.99|       4|1399.96|   297.4915|     MIDWEST|
|   O0004|     89.99|       2| 179.98|    85.4905|       SOUTH|
+--------+----------+--------+-------+-----------+------------+
only showing top 4 rows


**Task 4**

From customers.csv, select all columns except email and country using the list comprehension approach.

 How many columns does the result have?

In [5]:
customers=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv",header=True,inferSchema=True)
customers.select([column for column in customers.columns if column not in ['email','country']]).show(4)

+-----------+----------+---------+-----------+-----+-----------+----------+
|customer_id|first_name|last_name|       city|state|signup_date|   segment|
+-----------+----------+---------+-----------+-----+-----------+----------+
|       C001|     James| Anderson|   New York|   NY| 2021-03-15|Enterprise|
|       C002|     Maria|   Garcia|Los Angeles|   CA| 2021-05-22|       SMB|
|       C003|    Robert|  Johnson|    Chicago|   IL| 2020-11-08|Enterprise|
|       C004|     Linda| Martinez|    Houston|   TX| 2022-01-30|       SMB|
+-----------+----------+---------+-----------+-----+-----------+----------+
only showing top 4 rows


In [6]:
spark.stop()